In [1]:
GOLD = [
    {
        "method": "Chemical Vapor Deposition",
        "temperature": 773.15,
        "lab_id": "LAB-001",
        "description": "Deposited thin film at high temperature.",
        "observation": "Stable film with uniform coverage.",
        "outcome": "Success, target efficiency achieved.",
        "steps": [
            {"name": "deposit", "duration": 120},
            {"name": "anneal", "duration": 60},
        ],
    },
    {
        "method": "Sputtering",
        "temperature": 500.0,
        "lab_id": "LAB-002",
        "description": "Sputtered target onto substrate.",
        "observation": "Thin but patchy coating observed.",
        "outcome": "Partial success, some defects present.",
        "steps": [
            {"name": "clean", "duration": 30},
            {"name": "deposit", "duration": 90},
        ],
    },
    {
        # No lab_id, no description -> makes them optional in inferred schema
        "method": "Pulsed Laser Deposition",
        "temperature": 700.0,
        "observation": "Smooth crystalline film.",
        "outcome": "Fully successful.",
        "steps": [
            {"name": "ablate", "duration": 45},
            {"name": "deposit", "duration": 90},
        ],
    },
]

EXTRACTED = [
    {
        # method synonym, extra hallucinated step
        "method": "CVD",
        "temperature": 773.15,
        "lab_id": "LAB-001",
        "description": "Film deposited using CVD process.",
        "observation": "Film is stable with even coating.",          # paraphrase of gold
        "outcome": "Experiment succeeded, reached target efficiency.", # paraphrase of gold
        "steps": [
            {"name": "deposit", "duration": 120},
            {"name": "anneal", "duration": 60},
            {"name": "cool", "duration": 30},  # hallucinated
        ],
    },
    {
        # temperature wrong (450 vs 500), missing step, missing lab_id
        "method": "Sputtering",
        "temperature": 450.0,
        "description": "Standard sputtering procedure.",
        "observation": "Coating was thin and had visible patches.",  # paraphrase
        "outcome": "Film had major defects throughout.",              # NOT a paraphrase -- factual disagreement
        "steps": [
            {"name": "clean", "duration": 30},
            # missing "deposit" step -> omission
        ],
    },
    {
        # perfect match
        "method": "Pulsed Laser Deposition",
        "temperature": 700.0,
        "observation": "Smooth crystalline film.",   # exact match -> short-circuits
        "outcome": "Fully successful.",               # exact match -> short-circuits
        "steps": [
            {"name": "ablate", "duration": 45},
            {"name": "deposit", "duration": 90},
        ],
    },
]

In [2]:
from struct_extract_eval import infer_schema, generate_eval_schema, evaluate, validate_gold

resolved_schema = infer_schema(GOLD)
eval_schema = generate_eval_schema(schema=resolved_schema)
# manual edits to eval schema
# validate_gold(GOLD, eval_schema)

results = evaluate(EXTRACTED, GOLD, eval_schema)


In [3]:
print(f"  Records:        {results.total_records}")
print(f"  Fields scored:  {results.total_fields}")
print(f"  Precision:      {results.mean_precision:.3f}")
print(f"  Recall:         {results.mean_recall:.3f}")
print(f"  F1:             {results.mean_f1:.3f}")
print(f"  Omissions:      {results.total_omissions}")
print(f"  Hallucinations: {results.total_hallucinations}")

  Records:        3
  Fields scored:  30
  Precision:      0.633
  Recall:         0.643
  F1:             0.633
  Omissions:      2
  Hallucinations: 3


In [25]:
from struct_extract_eval.core.comparators.comparator import ComparatorResult
from typing import Any
from copy import deepcopy
from struct_extract_eval.core.comparators.registry import register, _registry

def always_match(gold: Any, extracted: Any, params: dict[str, Any]) -> ComparatorResult:

    return ComparatorResult(score=1.0, comparator="always_match", reason="always match")

_registry.pop("always_match", None)
register("always_match", always_match)
before_edit = deepcopy(eval_schema)
props = eval_schema["properties"]

before_semantic = deepcopy(eval_schema)
props["observation"]["x-eval-compare"] = "always_match"  # for testing: forces a match regardless of text
props["outcome"]["x-eval-compare"] = "always_match"  # for testing: forces a mismatch regardless of text

In [26]:
results = evaluate(EXTRACTED, GOLD, eval_schema)
print(f"  Records:        {results.total_records}")
print(f"  Fields scored:  {results.total_fields}")
print(f"  Precision:      {results.mean_precision:.3f}")
print(f"  Recall:         {results.mean_recall:.3f}")
print(f"  F1:             {results.mean_f1:.3f}")
print(f"  Omissions:      {results.total_omissions}")
print(f"  Hallucinations: {results.total_hallucinations}")


  Records:        3
  Fields scored:  30
  Precision:      0.767
  Recall:         0.794
  F1:             0.772
  Omissions:      2
  Hallucinations: 3
